In [0]:
########################
#Install Dependancies
########################
#Install openpyxl so can export as excel (only required to once for your computer)
%pip install openpyxl
%pip install pycountry
# dbutils.library.restartPython()

In [0]:
########################
#Setup for Tranformation Tests
########################

####################
this_notebook_state = "ARCHIVE_EVENTDATE"

#Setup Test Reporting
from datetime import datetime, timedelta
run_user = spark.sql("SELECT current_user()").first()[0]
run_tag = "Testing Transformation Tests"
run_by_automation_name = "Transformation_Tests"
#Capture Test Start datetime
run_start_datetime = datetime.now()
#####################

#Setup Notebook Parameters (Defaulting to Payment Pending and Running all tests)
dbutils.widgets.text("state_under_test", "")


# Read parameters
state_under_test = dbutils.widgets.get("state_under_test")

#Set Default Values if not called from another Notebook
if state_under_test == "" :
    state_under_test = this_notebook_state

#List of Fields to Exclude in Child Notebooks Called
child_fields_to_exclude = []

print(f"Testing State = {state_under_test}")

#Restart Python When needed (when changed tests)
# dbutils.library.restartPython()

#Import models.test_result as test_result
from models.test_result import TestResult

#Import asdict to convert Dataclass to Dictionary
from dataclasses import asdict
import os

#Setup Global Variables
all_test_results = []
current_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
base_path = current_path.rsplit("/", 1)[0] + "/"
#Below will be replaced eventually to store the reults in a spark table
test_results_path= "/Workspace/Users/" + run_user + "/Results/Transformation_Tests"
os.makedirs(test_results_path, exist_ok=True)


In [0]:
#Load Config and Setup Enviorment Variables
M2_bronze = None
M3_bronze = None
M3_silver = None
M6_bronze = None
bat = None
bhoref = None
bac = None
bll = None
b = None
M4_silver = None
M4_bronze = None
docsr = None
H_bronze = None
H_silver = None

config = spark.read.option("multiline", "true").json("dbfs:/configs/config.json")
env_name = config.first()["env"].strip().lower()
lz_key = config.first()["lz_key"].strip().lower()

KeyVault_name = f"ingest{lz_key}-meta002-{env_name}"

# Service principal credentials
client_id = dbutils.secrets.get(KeyVault_name, "SERVICE-PRINCIPLE-CLIENT-ID")
client_secret = dbutils.secrets.get(KeyVault_name, "SERVICE-PRINCIPLE-CLIENT-SECRET")
tenant_id = dbutils.secrets.get(KeyVault_name, "SERVICE-PRINCIPLE-TENANT-ID")

# Storage account names
curated_storage = f"ingest{lz_key}curated{env_name}"
checkpoint_storage = f"ingest{lz_key}xcutting{env_name}"
raw_storage = f"ingest{lz_key}raw{env_name}"
landing_storage = f"ingest{lz_key}landing{env_name}"
external_storage = f"ingest{lz_key}external{env_name}"

# Spark config for curated storage
spark.conf.set(f"fs.azure.account.auth.type.{curated_storage}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{curated_storage}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{curated_storage}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{curated_storage}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{curated_storage}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")

# Spark config for checkpoint storage
spark.conf.set(f"fs.azure.account.auth.type.{checkpoint_storage}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{checkpoint_storage}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{checkpoint_storage}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{checkpoint_storage}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{checkpoint_storage}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")

# Spark config for raw storage
spark.conf.set(f"fs.azure.account.auth.type.{raw_storage}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{raw_storage}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{raw_storage}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{raw_storage}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{raw_storage}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")

# Spark config for landing storage
spark.conf.set(f"fs.azure.account.auth.type.{landing_storage}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{landing_storage}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{landing_storage}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{landing_storage}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{landing_storage}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")

# Spark config for external storage
spark.conf.set(f"fs.azure.account.auth.type.{external_storage}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{external_storage}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{external_storage}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{external_storage}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{external_storage}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")

# Setting variables for use in subsequent cells
bronze_path = f"abfss://bronze@ingest{lz_key}curated{env_name}.dfs.core.windows.net/ARIADM/ARM/"
silver_path = f"abfss://silver@ingest{lz_key}curated{env_name}.dfs.core.windows.net/ARIADM/ARM/"
gold_path = f"abfss://gold@ingest{lz_key}curated{env_name}.dfs.core.windows.net/ARIADM/ARM/"

raw_mnt = f"abfss://raw@ingest{lz_key}raw{env_name}.dfs.core.windows.net/ARIADM/ARM/APPEALS/{AppealCategory}"
landing_mnt =  f"abfss://landing@ingest{lz_key}landing{env_name}.dfs.core.windows.net/SQLServer/Sales/IRIS/dbo/"

test_from_state = "Archive"

from pyspark.sql.functions import (
    col, when, lit, array, struct, collect_list,
    max as spark_max, date_format, row_number, expr,
    size, udf, coalesce, concat_ws, concat, trim, year, split, datediff,
    collect_set, current_timestamp,transform, first, array_contains, current_date
)

from pyspark.sql.window import Window
import pyspark.sql.functions as F
from models.test_result import TestResult
import re
import inspect
import time as perf_time

In [0]:
combos_Appeal_FT = [
    (10,0), (10,2), (10,13), (10,25), (10,39), (10,40), (10,80), (10,82), (10,109), (10,119),
    (10,120), (10,121), (10,122), (26,0), (26,1), (26,2), (26,13), (26,25), (26,27), (26,39),
    (26,40), (26,50), (26,52), (26,77), (26,80), (26,89), (30,1), (30,2), (30,13), (30,25),
    (30,50), (30,72), (30,80), (31,1), (31,2), (31,13), (31,25), (31,80), (35,1), (35,2),
    (35,10), (35,12), (35,13), (35,15), (35,16), (35,25), (35,27), (35,29), (35,45), (35,56),
    (35,57), (35,81), (36,0), (36,1), (36,2), (36,27), (36,40), (36,50), (37,0), (37,1),
    (37,2), (37,5), (37,13), (37,14), (37,25), (37,27), (37,37), (37,39), (37,40), (37,50),
    (37,52), (37,72), (37,77), (37,80), (37,86), (37,89), (37,125), (38,0), (38,1), (38,2),
    (38,5), (38,13), (38,14), (38,25), (38,27), (38,39), (38,40), (38,50), (38,72), (38,80),
    (38,89), (38,125), (39,0), (39,14), (39,25), (39,30), (39,31), (39,80), (39,86), (42,86),
    (46,1), (46,25), (46,31), (46,50), (46,86), (50,91), (51,0), (51,93), (51,94), (52,0),
    (52,91), (52,95)
]

combos_Appeal_UT = [
    (27,0), (27,1), (27,2), (27,25), (27,50), (27,74), (27,80), (27,90), (28,0), (28,1),
    (28,2), (28,25), (28,50), (28,80), (29,14), (29,25), (29,50), (29,63), (29,80), (32,1),
    (32,2), (32,13), (32,25), (32,50), (32,80), (33,1), (33,2), (33,13), (33,25), (33,80),
    (34,0), (34,25), (34,30), (34,31), (34,50), (34,62), (34,79), (34,80), (40,0), (40,14),
    (40,25), (40,30), (40,31), (41,30), (41,31), (42,0), (42,1), (42,2), (42,5), (42,13),
    (42,25), (42,37), (42,50), (42,80), (43,1), (43,2), (43,5), (43,25), (43,27), (43,50),
    (43,52), (43,80), (44,0), (44,1), (44,27), (44,31), (44,50), (44,80), (45,0), (45,25),
    (45,30), (45,31), (45,50)
]

combos_Bail = [
    (4,0), (4,1), (4,2), (4,4), (4,6), (4,7), (4,8), (4,9), (4,23), (4,24),
    (4,25), (4,30), (4,31), (4,32), (4,33), (4,34), (4,49), (4,50), (4,60), (4,96),
    (4,97), (4,100), (4,101), (4,102), (4,103), (4,107), (4,110), (4,112), (4,114), (4,115),
    (4,117), (4,123), (4,124), (6,0), (6,9), (6,11), (6,25), (6,30), (6,31), (6,33),
    (6,41), (6,49), (6,50), (6,60), (11,9), (11,11), (11,30), (11,31), (11,33), (11,41),
    (11,50), (11,60), (18,0), (18,4), (18,6), (18,7), (18,8), (18,9), (18,11), (18,23),
    (18,24), (18,25), (18,30), (18,31), (18,32), (18,33), (18,41), (18,49), (18,50), (18,118),
    (18,123), (18,124), (19,0), (19,7), (19,8), (19,9), (19,24), (19,25), (19,30), (19,31),
    (19,32), (19,33), (19,49), (19,50), (19,102), (19,115), (19,117), (19,123), (19,124), (35,2),
    (35,4), (35,6), (35,7), (35,9), (35,11), (35,23), (35,24), (35,25), (35,30), (35,31),
    (35,32), (35,33), (35,34), (35,35), (35,36), (35,41), (35,49), (35,60)
]

valid_combos_Appeal_FT_condition = struct(
    col("CaseStatus").cast("int"),
    col("Outcome").cast("int")
).isin(
    [struct(lit(cs), lit(out)) for cs, out in combos_Appeal_FT]
)

valid_combos_Appeal_UT_condition = struct(
    col("CaseStatus").cast("int"),
    col("Outcome").cast("int")
).isin(
    [struct(lit(cs), lit(out)) for cs, out in combos_Appeal_UT]
)

valid_combos_Bail_condition = struct(
    col("CaseStatus").cast("int"),
    col("Outcome").cast("int")
).isin(
    [struct(lit(cs), lit(out)) for cs, out in combos_Bail]
)

In [0]:
all_test_results = []
perf_timings = []

In [0]:
t0 = perf_time.time()

bronze = spark.read.table("hive_metastore.ariadm_arm_fta.bronze_appealcase_status_sc_ra_cs").select("CaseNo", "DecisionDate", "CaseStatus", "StatusId", "Outcome")

silver = spark.read.table("hive_metastore.ariadm_arm_fta.silver_archive_metadata").select("client_identifier", "event_date")
# to test next states, i just need to change where we get the silver data from (FTA, UTA, BAILS, SBAILS).
# for FPA, TD - current date
# for JOH - current date + 7 yrs

event_date1_df = bronze.join(
    silver,
    bronze["CaseNo"] == silver["client_identifier"],
    "inner"
).drop("client_identifier")

# event_date1_df.display()

# if Outcome = 38 & CaseStatus = 4 OR Outcome = 38 & CaseStatus = 19
# then eventDate = decision_date_prev (2nd latest date)
# else if Outcome & CaseStatus is valid, eventDate = decision_date (latest date)
# else eventDate = currentDate

# Rank decision dates per case (latest first)
w = Window.partitionBy("CaseNo").orderBy(col("StatusId").desc())

ranked_df = event_date1_df.withColumn(
    "rn",
    row_number().over(w)
).filter(
    col("rn") <= 2
)

# Get latest and second latest decision dates
decision_dates = (
        ranked_df
        .groupBy("CaseNo")
        .agg(
            F.max(F.when(col("rn") == 1, col("DecisionDate"))).alias("decision_date_latest"),
            F.max(F.when(col("rn") == 2, col("DecisionDate"))).alias("decision_date_prev"),
            F.max(F.when(col("rn") == 1, col("CaseStatus"))).alias("CaseStatus_Ranked"),
            F.max(F.when(col("rn") == 1, col("Outcome"))).alias("Outcome_Ranked")
        )
    )

test_df = event_date1_df.join(
    decision_dates,
    (event_date1_df["CaseStatus"] == decision_dates["CaseStatus_Ranked"]) & 
    (event_date1_df["CaseNo"] == decision_dates["CaseNo"]) & 
    (event_date1_df["Outcome"] == decision_dates["Outcome_Ranked"]),
    "inner"
)

# Apply business logic
result = test_df.withColumn(
    "expected_eventDate",
    when(
        (col("Outcome") == 38) &
        (col("CaseStatus").isin(4, 19)),
        col("decision_date_prev")
    ).when(
        valid_combos_Appeal_FT_condition,
        col("decision_date_latest")
    ).otherwise(
        current_date()
    )
)

result = result.filter(
    col("event_date") != col("expected_eventDate")
)

result.display()

if result.count() != 0:
    FTA_result = TestResult("FTA", "FAIL", f"FTA acceptance criteria failed: {str(result.count())} cases have been found where eventDate is not as expected.", test_from_state, inspect.stack()[0].function)
else:
    FTA_result = TestResult("FTA", "PASS", f"FTA acceptance criteria passed: all cases have the expected eventDate.", test_from_state, inspect.stack()[0].function)

all_test_results.append(FTA_result)

dur = perf_time.time() - t0

perf_timings.append({"test_name": "FTA eventDate", "test_from_state": "FTA", "elapsed_seconds": dur, "result_count": test_df.count()})

In [0]:
t0 = perf_time.time()

bronze = spark.read.table("hive_metastore.ariadm_arm_uta.bronze_appealcase_status_sc_ra_cs").select("CaseNo", "DecisionDate", "CaseStatus", "StatusId", "Outcome")

silver = spark.read.table("hive_metastore.ariadm_arm_uta.silver_archive_metadata").select("client_identifier", "event_date")
# to test next states, i just need to change where we get the silver data from (FTA, UTA, BAILS, SBAILS).
# for FPA, TD - current date
# for JOH - current date + 7 yrs

event_date1_df = bronze.join(
    silver,
    bronze["CaseNo"] == silver["client_identifier"],
    "inner"
).drop("client_identifier")

# event_date1_df.display()

# if Outcome = 38 & CaseStatus = 4 OR Outcome = 38 & CaseStatus = 19
# then eventDate = decision_date_prev (2nd latest date)
# else if Outcome & CaseStatus is valid, eventDate = decision_date (latest date)
# else eventDate = currentDate

# Rank decision dates per case (latest first)
w = Window.partitionBy("CaseNo").orderBy(col("StatusId").desc())

ranked_df = event_date1_df.withColumn(
    "rn",
    row_number().over(w)
).filter(
    col("rn") <= 2
)

# Get latest and second latest decision dates
decision_dates = (
        ranked_df
        .groupBy("CaseNo")
        .agg(
            F.max(F.when(col("rn") == 1, col("DecisionDate"))).alias("decision_date_latest"),
            F.max(F.when(col("rn") == 2, col("DecisionDate"))).alias("decision_date_prev"),
            F.max(F.when(col("rn") == 1, col("CaseStatus"))).alias("CaseStatus_Ranked"),
            F.max(F.when(col("rn") == 1, col("Outcome"))).alias("Outcome_Ranked")
        )
    )

test_df = event_date1_df.join(
    decision_dates,
    (event_date1_df["CaseStatus"] == decision_dates["CaseStatus_Ranked"]) & 
    (event_date1_df["CaseNo"] == decision_dates["CaseNo"]) & 
    (event_date1_df["Outcome"] == decision_dates["Outcome_Ranked"]),
    "inner"
)

# Apply business logic
result = test_df.withColumn(
    "expected_eventDate",
    when(
        (col("Outcome") == 38) &
        (col("CaseStatus").isin(4, 19)),
        col("decision_date_prev")
    ).when(
        valid_combos_Appeal_UT_condition,
        col("decision_date_latest")
    ).otherwise(
        current_date()
    )
)

result = result.filter(
    col("event_date") != col("expected_eventDate")
)

result.display()

if result.count() != 0:
    UTA_result = TestResult("UTA", "FAIL", f"UTA acceptance criteria failed: {str(result.count())} cases have been found where eventDate is not as expected.", test_from_state, inspect.stack()[0].function)
else:
    UTA_result = TestResult("UTA", "PASS", f"UTA acceptance criteria passed: all cases have the expected eventDate.", test_from_state, inspect.stack()[0].function)

all_test_results.append(UTA_result)

dur = perf_time.time() - t0

perf_timings.append({"test_name": "UTA eventDate", "test_from_state": "UTA", "elapsed_seconds": dur, "result_count": test_df.count()})

In [0]:
t0 = perf_time.time()

bronze = spark.read.table("hive_metastore.aria_bails.bronze_bail_status_sc_ra_cs").select("CaseNo", "DecisionDate", "CaseStatus", "StatusId", "Outcome")

silver = spark.read.table("hive_metastore.aria_bails.silver_bail_meta_data").select("client_identifier", "event_date")
# to test next states, i just need to change where we get the silver data from (FTA, UTA, BAILS, SBAILS).
# for FPA, TD - current date
# for JOH - current date + 7 yrs

event_date1_df = bronze.join(
    silver,
    bronze["CaseNo"] == silver["client_identifier"],
    "inner"
).drop("client_identifier")

# event_date1_df.display()

# if Outcome = 38 & CaseStatus = 4 OR Outcome = 38 & CaseStatus = 19
# then eventDate = decision_date_prev (2nd latest date)
# else if Outcome & CaseStatus is valid, eventDate = decision_date (latest date)
# else eventDate = currentDate

# Rank decision dates per case (latest first)
w = Window.partitionBy("CaseNo").orderBy(col("StatusId").desc())

ranked_df = event_date1_df.withColumn(
    "rn",
    row_number().over(w)
).filter(
    col("rn") <= 2
)

# Get latest and second latest decision dates
decision_dates = (
        ranked_df
        .groupBy("CaseNo")
        .agg(
            F.max(F.when(col("rn") == 1, col("DecisionDate"))).alias("decision_date_latest"),
            F.max(F.when(col("rn") == 2, col("DecisionDate"))).alias("decision_date_prev"),
            F.max(F.when(col("rn") == 1, col("CaseStatus"))).alias("CaseStatus_Ranked"),
            F.max(F.when(col("rn") == 1, col("Outcome"))).alias("Outcome_Ranked")
        )
    )

test_df = event_date1_df.join(
    decision_dates,
    (event_date1_df["CaseStatus"] == decision_dates["CaseStatus_Ranked"]) & 
    (event_date1_df["CaseNo"] == decision_dates["CaseNo"]) & 
    (event_date1_df["Outcome"] == decision_dates["Outcome_Ranked"]),
    "inner"
)

# Apply business logic
result = test_df.withColumn(
    "expected_eventDate",
    when(
        (col("Outcome") == 38) &
        (col("CaseStatus").isin(4, 19)),
        col("decision_date_prev")
    ).when(
        valid_combos_Bail_condition,
        col("decision_date_latest")
    ).otherwise(
        current_date()
    )
)

result = result.filter(
    col("event_date") != col("expected_eventDate")
)

result.display()

if result.count() != 0:
    Bails_result = TestResult("Bails", "FAIL", f"Bails acceptance criteria failed: {str(result.count())} cases have been found where eventDate is not as expected.", test_from_state, inspect.stack()[0].function)
else:
    Bails_result = TestResult("Bails", "PASS", f"Bails acceptance criteria passed: all cases have the expected eventDate.", test_from_state, inspect.stack()[0].function)

all_test_results.append(Bails_result)

dur = perf_time.time() - t0

perf_timings.append({"test_name": "Bails eventDate", "test_from_state": "Bails", "elapsed_seconds": dur, "result_count": test_df.count()})

In [0]:
########################
# PROCESS RESULTS
########################
from collections import defaultdict
from Test_Functions.report_helpers import (
    count_by_status,
    create_run, create_results, create_perf_results,
    build_report_folder,
    write_csv, write_xlsx, write_html,
)

# ---- 1. COUNTS (visible, shared with helpers) ----
counts = count_by_status(all_test_results)

# ---- 2. CONSOLE SUMMARY (visible, easy to tweak) ----
grouped = defaultdict(list)
for r in all_test_results:
    grouped[r.test_from_state].append(r)

lines = []
lines.append("=" * 80)
lines.append(f"TEST EXECUTION REPORT -- {state_under_test}")
lines.append("-" * 80)
lines.append(f"Total tests : {counts['TOTAL']}")
lines.append(f"Passed      : {counts['PASS']}")
lines.append(f"Failed      : {counts['FAIL']}")
lines.append(f"No Data     : {counts['NO_DATA']}")
lines.append(f"Error       : {counts['ERROR']}")
lines.append("-" * 80)

for state in sorted(grouped):
    c = count_by_status(grouped[state])
    lines.append(f"\nSTATE: {state}")
    lines.append(f"Tests: {c['TOTAL']} | Passed: {c['PASS']} | Failed: {c['FAIL']} | No Data: {c['NO_DATA']} | Error: {c['ERROR']}")
    lines.append("-" * 60)
print("\n".join(lines))
# ---- 2b. PERFORMANCE TIMINGS ----
if perf_timings:
    lines2 = []
    lines2.append("")
    lines2.append("=" * 80)
    lines2.append("PERFORMANCE TIMINGS")
    lines2.append("-" * 80)
    total_secs = 0.0
    for pt in perf_timings:
        total_secs += pt["elapsed_seconds"]
        lines2.append(f"  {pt['test_name']:65s} {int(pt['elapsed_seconds']//3600)}h {int(pt['elapsed_seconds']%3600//60)}m {int(pt['elapsed_seconds']%60)}s ({pt['elapsed_seconds']:.1f}s)  {pt['result_count']} results")
    lines2.append("-" * 80)
    lines2.append(f"  {'TOTAL':65s} {int(total_secs//3600)}h {int(total_secs%3600//60)}m {int(total_secs%60)}s ({total_secs:.1f}s)")
    lines2.append("=" * 80)
    print("\n".join(lines2))


# ---- 3. IF THIS IS THE TOP STATE: WRITE FILES + TABLES ----
if state_under_test == this_notebook_state:
    # Build ONE folder for this run so all 3 files land together
    folder, timestamp, _ = build_report_folder(test_results_path, state_under_test)
    print(f"Reports folder: {folder}")

    try:
        csv_path = write_csv(all_test_results, state_under_test, folder, timestamp)
        print(f"CSV  : {csv_path}")
    except Exception as e:
        print(f"CSV  write failed: {e}")

    try:
        xlsx_path = write_xlsx(all_test_results, state_under_test, folder, timestamp)
        print(f"XLSX : {xlsx_path}")
    except Exception as e:
        print(f"XLSX write failed: {e}")

    try:
        html_path = write_html(all_test_results, state_under_test, folder, timestamp, counts=counts)
        print(f"HTML : {html_path}")
    except Exception as e:
        print(f"HTML write failed: {e}")

    # Spark tables
    run_status = "PASS" if counts["FAIL"] == 0 and counts["PASS"] >= 1 else "FAIL"
    try:
        run_id = create_run(
            spark,
            run_user=run_user,
            run_start_datetime=run_start_datetime,
            run_by_automation_name=run_by_automation_name,
            run_tag=run_tag,
            run_status=run_status,
            state_under_test=state_under_test,
        )
        print(f"Created run_id={run_id}")
    except Exception as e:
        print(f"create_run FAILED: {e}")
        run_id = None

    if run_id:
        try:
            n = create_results(spark, run_id, all_test_results)
            print(f"Saved {n} result rows")
        except Exception as e:
            print(f"create_results FAILED: {e}")

        try:
            np = create_perf_results(spark, run_id, perf_timings, state_under_test)
            print(f"Saved {np} perf rows")
        except Exception as e:
            print(f"create_perf_results FAILED: {e}")

else:
    # Chain child â€” return data to parent
    import json
    from dataclasses import asdict
    all_test_results_string = json.dumps([asdict(r) for r in all_test_results])
    print(f"Exiting Notebook : {this_notebook_state} and returning Data to Parent notebook : {state_under_test}")
    dbutils.notebook.exit(all_test_results_string)